In [1]:
pip install pyspark

In [2]:
import pyspark

**Understanding Pivot and Unpivot in PySpark DataFrames**

Pivot and Unpivot are powerful operations that help reshape DataFrames. They are widely used in data analytics, especially in reporting and dashboarding.

# 1️⃣ What is Pivot in PySpark?

**Pivoting is the process of transforming rows into columns based on the values of a specific column.**

**📌 Use Case: You have a dataset of student marks in different subjects, and you want to convert it into a student-wise table where each subject becomes a column.**

In [3]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *

spark = SparkSession.builder.appName("PivotExample").getOrCreate()

In [4]:
data = [
    ("Rahul", "Maths", 40),
    ("Rahul", "Maths", 50),
    ("Rahul", "Science", 35),
    ("Priya", "Maths", 45),
    ("Priya", "Science", 40),
    ("Priya", "Science", 42),
    ("Amit", "Maths", 30),
    ("Amit", "Science", 25),
    ("Amit", "Science", 30),
]

df = spark.createDataFrame(data, ["Name", "Subject", "Marks"])
df.show()

+-----+-------+-----+
| Name|Subject|Marks|
+-----+-------+-----+
|Rahul|  Maths|   40|
|Rahul|  Maths|   50|
|Rahul|Science|   35|
|Priya|  Maths|   45|
|Priya|Science|   40|
|Priya|Science|   42|
| Amit|  Maths|   30|
| Amit|Science|   25|
| Amit|Science|   30|
+-----+-------+-----+



In [ ]:
# Example 1: Student Marks Pivot

### Example 1: Student Marks Pivot

In [ ]:

df.groupBy("")

**✔ Transformed long format into wide format using pivot.**

### Example 2: Monthly Sales Pivot (Retail Dataset)

**Use Case: find the total sale per month for each platform/company**

In [ ]:
data = [
    ("Amazon", "Jan", 10000),
    ("Amazon", "Feb", 20000),
    ("Amazon", "Jan", 5000),
    ("Flipkart", "Jan", 8000),
    ("Flipkart", "Feb", 12000),
]

df = spark.createDataFrame(data, ["Platform", "Month", "Sales"])
df.show()


+--------+-----+-----+
|Platform|Month|Sales|
+--------+-----+-----+
|  Amazon|  Jan|10000|
|  Amazon|  Feb|20000|
|  Amazon|  Jan| 5000|
|Flipkart|  Jan| 8000|
|Flipkart|  Feb|12000|
+--------+-----+-----+



In [ ]:
pivot_df = df.groupBy("Platform").pivot("Month").agg(sum("Sales"))
pivot_df.show()

# This is a common scenario in sales reporting.

+--------+-----+-----+
|Platform|  Feb|  Jan|
+--------+-----+-----+
|Flipkart|12000| 8000|
|  Amazon|20000|15000|
+--------+-----+-----+



# 2️⃣ What is Unpivot in PySpark?

**Unpivoting is the reverse of pivoting. It transforms columns into rows. PySpark does not have a direct unpivot() function, but it can be done using selectExpr() and stack().**

**Use Case: You want to convert student marks (wide format) back into a long format for analysis or visualization.**

### Example 1: Unpivot Student Marks

In [ ]:
# Step 1: Create Simple Wide Format Data

data = [
    ("Amit", 80, 70),
    ("Priya", 90, 85),
    ("Raj", 75, 60)
]
columns = ["Name", "Math", "Science"]
df_marks = spark.createDataFrame(data, columns)
df_marks.show()

+-----+----+-------+
| Name|Math|Science|
+-----+----+-------+
| Amit|  80|     70|
|Priya|  90|     85|
|  Raj|  75|     60|
+-----+----+-------+



In [ ]:
# Step 2: Unpivot (Convert Columns → Rows)

In [ ]:
unpivot_df = df_marks.selectExpr(
    "Name",
    "stack(2, 'Math', Math, 'Science', Science) as (Subject, Marks)"
)

unpivot_df.show()

+-----+-------+-----+
| Name|Subject|Marks|
+-----+-------+-----+
| Amit|   Math|   80|
| Amit|Science|   70|
|Priya|   Math|   90|
|Priya|Science|   85|
|  Raj|   Math|   75|
|  Raj|Science|   60|
+-----+-------+-----+



### Example 2

In [ ]:
data = [
    ("Ravi", 1, 0),
    ("Sneha", 1, 1),
    ("Amit", 0, 1)
]

columns = ["Name", "Day1", "Day2"]

df_attendance = spark.createDataFrame(data, columns)
df_attendance.show()

# Here: 1 = Present, 0 = Absent

+-----+----+----+
| Name|Day1|Day2|
+-----+----+----+
| Ravi|   1|   0|
|Sneha|   1|   1|
| Amit|   0|   1|
+-----+----+----+



In [ ]:
# Unpivot Day Columns into Rows
unpivot_df = df_attendance.selectExpr(
    "Name",
    "stack(2, 'Day1', Day1, 'Day2', Day2) as (Day, Status)"
)

unpivot_df.show()


+-----+----+------+
| Name| Day|Status|
+-----+----+------+
| Ravi|Day1|     1|
| Ravi|Day2|     0|
|Sneha|Day1|     1|
|Sneha|Day2|     1|
| Amit|Day1|     0|
| Amit|Day2|     1|
+-----+----+------+



### What is stack() in PySpark?

**stack() is a SQL expression function that is used to unpivot columns into rows.
It's like flipping your table vertically for selected columns.**

In [ ]:
# syntax
'''
stack(n, col_name1, value1, col_name2, value2, ...) AS (new_col1, new_col2)

'''

In [ ]:
stack(2,         # 2 rows will be created for each row (Day1 and Day2) # 2 columns will be convert into  rows
      'Day1', Day1,  # First row → label 'Day1', value from column Day1
      'Day2', Day2   # Second row → label 'Day2', value from column Day2
) as (Day, Status)   # Day → column name (string), Status → actual value (0 or 1)


In [ ]:
#Syntax Breakdown:
'''
n = Number of rows to be created per original row (i.e., how many columns you're unpivoting)

The rest are in pairs: 'column_name', column_value

You give new column names at the end using AS (col1, col2)
'''

**Pivot = "Many Rows → Few Columns" (Summary table format)**

**Unpivot = "Many Columns → Long Rows" (Ideal for visualizations, long format)**